# Quantum Rademacher Complexity with Finite-Shot Corrections: Standalone Experiments

This notebook reproduces and improves the manuscript's Section 5 experiments for:

> *Rademacher Complexity of Quantum Hypothesis Classes: Generalization Bounds with Finite-Shot Corrections*

**Scope and caveats:**
- **Experiment 0** (Direct H_POVM Rademacher simulation) is **theorem-aligned** for Proposition 3.1. It uses the exact positive-eigenvalue supremum formula from the proof.
- **Experiments 1–4** (VQC experiments) use a **restricted variational circuit class** and are structured sanity checks, **not** worst-case validation of the full H_POVM class.
- VQC training uses **noiseless simulator gradients**. Finite-shot effects are introduced only during **evaluation**. The VQC experiments therefore test post-training finite-shot risk bounds for fixed learned hypotheses, not finite-shot optimization dynamics.
- The **primary empirical gap** uses the Lipschitz score loss ℓ(p,y) = |p−y|, which is bounded in [0,1] and 1-Lipschitz in p.
- Hard-threshold accuracy is reported as an **auxiliary diagnostic only** and is **not** used for theorem validity.

In [1]:
import os, sys, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from sklearn.datasets import make_moons, make_circles, make_classification, make_blobs
from sklearn.preprocessing import StandardScaler
import pennylane as qml
from tqdm.auto import tqdm
import joblib
warnings.filterwarnings('ignore')

# ─── Global controls ──────────────────────────────────────────────────────────
FAST_MODE    = True
SMOKE_TEST   = False
GLOBAL_SEED  = 42
DELTA        = 0.05
L_LIPSCHITZ  = 1.0
EPS          = 1e-12
TIGHTNESS_GAP_MIN = 1e-3
S_TEST_LARGE = 2000

if SMOKE_TEST:
    N_TEST_LARGE    = 100
    EXP1_SEEDS      = {4: 2, 16: 2}
    EXP2_SEEDS      = {4: 2, 16: 2}
    EXP3_SEEDS      = 2
    EXP4_SEEDS      = 2
    HPOVM_N_SIGMA   = 20
    HPOVM_STATE_REPEATS = 2
    N_STEPS         = 5
elif FAST_MODE:
    N_TEST_LARGE    = 500
    EXP1_SEEDS      = {4: 6, 16: 4}
    EXP2_SEEDS      = {4: 6, 16: 4}
    EXP3_SEEDS      = 4
    EXP4_SEEDS      = 6
    HPOVM_N_SIGMA   = 200
    HPOVM_STATE_REPEATS = 5
    N_STEPS         = 10
else:
    N_TEST_LARGE    = 3000
    EXP1_SEEDS      = {4: 12, 16: 8}
    EXP2_SEEDS      = {4: 12, 16: 8}
    EXP3_SEEDS      = 8
    EXP4_SEEDS      = 12
    HPOVM_N_SIGMA   = 1000
    HPOVM_STATE_REPEATS = 20
    N_STEPS         = 50

# ─── Seeds ────────────────────────────────────────────────────────────────────
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# ─── Folders ──────────────────────────────────────────────────────────────────
for folder in ['figures', 'tables', 'results']:
    os.makedirs(folder, exist_ok=True)

# ─── Device preference ────────────────────────────────────────────────────────
def get_device(n_qubits):
    try:
        dev = qml.device('lightning.qubit', wires=n_qubits)
        return dev, 'lightning.qubit'
    except Exception:
        dev = qml.device('default.qubit', wires=n_qubits)
        return dev, 'default.qubit'

# ─── Library versions ─────────────────────────────────────────────────────────
print(f"numpy      : {np.__version__}")
print(f"pennylane  : {qml.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"matplotlib : {matplotlib.__version__}")
print(f"joblib     : {joblib.__version__}")
import sklearn; print(f"sklearn    : {sklearn.__version__}")
import tqdm as _tqdm; print(f"tqdm       : {_tqdm.__version__}")
import scipy; print(f"scipy      : {scipy.__version__}")
_, dev_name = get_device(2)
print(f"PennyLane device: {dev_name}")
print(f"\nFAST_MODE={FAST_MODE}, SMOKE_TEST={SMOKE_TEST}, GLOBAL_SEED={GLOBAL_SEED}")

numpy      : 2.2.6
pennylane  : 0.42.3
pandas     : 2.3.3
matplotlib : 3.10.7
joblib     : 1.5.2
sklearn    : 1.7.2
tqdm       : 4.67.1
scipy      : 1.15.3
PennyLane device: lightning.qubit

FAST_MODE=True, SMOKE_TEST=False, GLOBAL_SEED=42


## 3. Theory Functions

Exact theorem constants from Theorem 3.1:

$$R_\ell(M) \leq \hat{R}_{n,S,\ell}(M) + 2L\sqrt{\frac{d}{n}} + L\sqrt{\frac{\log(4n/\delta)}{2S}} + \sqrt{\frac{\log(2/\delta)}{2n}}$$

In [2]:
# ─── Theory functions ─────────────────────────────────────────────────────────

def absolute_score_loss(p1, y):
    return np.abs(p1 - y)          # L = 1, bounded in [0,1]

def brier_score_loss(p1, y):
    return (p1 - y) ** 2           # L <= 2 on [0,1], bounded in [0,1]

def sample_term(n, d, L=1.0):
    """Classical Rademacher term: 2L sqrt(d/n)."""
    return 2 * L * np.sqrt(d / n)

def shot_term_exact(n, S, delta=0.05, L=1.0):
    """Finite-shot term (exact): L sqrt(log(4n/delta)/(2S))."""
    return L * np.sqrt(np.log(4 * n / delta) / (2 * S))

def confidence_term_exact(n, delta=0.05):
    """McDiarmid confidence term: sqrt(log(2/delta)/(2n))."""
    return np.sqrt(np.log(2 / delta) / (2 * n))

def theorem_bound_exact(n, S, d, delta=0.05, L=1.0):
    """Full theorem 3.1 bound (exact constants)."""
    return (sample_term(n, d, L) +
            shot_term_exact(n, S, delta, L) +
            confidence_term_exact(n, delta))

def dominant_bound(n, S, d, delta=0.05, L=1.0):
    """Dominant two terms (sample + shot), omitting confidence term."""
    return sample_term(n, d, L) + shot_term_exact(n, S, delta, L)

def n_star_approx(B, d, delta=0.05):
    """Continuous approximate optimal n* = 2 sqrt(2dB / log(2B/delta))."""
    return 2 * np.sqrt(2 * d * B / np.log(2 * B / delta))

def S_star_approx(B, d, delta=0.05):
    """Continuous approximate optimal S* = B / n*."""
    return B / n_star_approx(B, d, delta)

def nearest_integer_allocation(B, d, delta=0.05):
    """Round n* to nearest integer, clip to [1,B], set S=max(1, B//n)."""
    n_cont = n_star_approx(B, d, delta)
    n_int = int(np.clip(np.round(n_cont), 1, B))
    S_int = max(1, B // n_int)
    return n_int, S_int

def safe_tightness_ratio(bound, gap, min_gap=1e-3):
    """Return bound/gap or nan if gap is too small."""
    if gap < min_gap:
        return np.nan
    return bound / gap

# ─── Theoretical tradeoff G(n) under fixed budget B ───────────────────────────
def G_tradeoff(n_arr, B, d, delta=0.05, L=1.0):
    """G(n) = 2L sqrt(d/n) + L sqrt(log(4n/delta)/(2*(B/n))) + sqrt(log(2/delta)/(2n))"""
    S_arr = np.maximum(1, B / n_arr)
    return theorem_bound_exact(n_arr, S_arr, d, delta, L)

# ─── Sanity-check allocation table ────────────────────────────────────────────
rows = []
for B in [500, 1000, 2000, 5000]:
    for d in [4, 16]:
        nc  = n_star_approx(B, d, DELTA)
        Sc  = S_star_approx(B, d, DELTA)
        ni, Si = nearest_integer_allocation(B, d, DELTA)
        # st  = sample_term(nc, d)
        # sht = shot_term_exact(nc, Sc, DELTA)
        # ct  = confidence_term_exact(nc, DELTA)
        st  = sample_term(ni, d)               # ← ni (integer) ✓
        sht = shot_term_exact(ni, Si, DELTA)   # ← ni, Si (integer) ✓
        ct  = confidence_term_exact(ni, DELTA) # ← ni (integer) ✓
        fb  = st + sht + ct
        rows.append(dict(B=B, d=d, n_star_cont=round(nc,2), S_star_cont=round(Sc,2),
                         n_star_int=ni, S_star_int=Si,
                         sample_term=round(st,4), shot_term=round(sht,4),
                         confidence_term=round(ct,4), full_bound=round(fb,4)))
alloc_df = pd.DataFrame(rows)
print(alloc_df.to_string(index=False))
alloc_df.to_csv('tables/theoretical_allocation_table.csv', index=False)

   B  d  n_star_cont  S_star_cont  n_star_int  S_star_int  sample_term  shot_term  confidence_term  full_bound
 500  4        40.19        12.44          40          12       0.6325     0.5799           0.2147      1.4271
 500 16        80.39         6.22          80           6       0.8944     0.8546           0.1518      1.9009
1000  4        54.95        18.20          55          18       0.5394     0.4827           0.1831      1.2052
1000 16       109.91         9.10         110           9       0.7628     0.7103           0.1295      1.6026
2000  4        75.29        26.56          75          26       0.4619     0.4090           0.1568      1.0277
2000 16       150.58        13.28         151          13       0.6510     0.6013           0.1105      1.3628
5000  4       114.49        43.67         114          43       0.3746     0.3256           0.1272      0.8274
5000 16       228.98        21.84         229          21       0.5287     0.4834           0.0897      1.1018


## 4. Experiment 0: Direct H_POVM Rademacher Simulation

**Theorem-aligned.** Implements the exact positive-eigenvalue supremum formula:
$$\sup_{0 \leq M \leq I} \text{Tr}(M A_\sigma) = \sum_{\lambda_j(A_\sigma) > 0} \lambda_j(A_\sigma)$$
where $A_\sigma = \frac{1}{n}\sum_i \sigma_i \rho_i$.

In [3]:
# ─── Experiment 0: Direct H_POVM Rademacher simulation ───────────────────────

def sample_random_density_matrix(d, rng):
    """Complex Ginibre construction for a random mixed state."""
    X = rng.standard_normal((d, d)) + 1j * rng.standard_normal((d, d))
    rho = X @ X.conj().T
    rho /= np.trace(rho)
    return rho

def hpovm_supremum_exact(A_sigma):
    """sum of positive eigenvalues of Hermitian A_sigma (real eigenvalues)."""
    eigs = np.linalg.eigvalsh(A_sigma)
    return float(np.sum(eigs[eigs > 0]))

def compute_hpovm_rademacher_one_sigma(rhos, n, rng):
    """For one Rademacher vector sigma, compute sup Tr(M A_sigma)."""
    sigma = rng.choice([-1, 1], size=n)
    A_sigma = np.tensordot(sigma, rhos, axes=([0],[0])) / n
    return hpovm_supremum_exact(A_sigma)

def hpovm_rademacher_estimate(d, n, n_sigma, n_state_repeats, seed):
    """Estimate empirical Rademacher complexity, return (mean, std)."""
    rng = np.random.default_rng(seed)
    all_estimates = []
    for _ in range(n_state_repeats):
        rhos = np.array([sample_random_density_matrix(d, rng) for _ in range(n)])
        vals = [compute_hpovm_rademacher_one_sigma(rhos, n, rng) for _ in range(n_sigma)]
        all_estimates.append(np.mean(vals))
    return float(np.mean(all_estimates)), float(np.std(all_estimates))

d_values = [4, 8, 16]
n_values_hpovm = [10, 20, 40, 80, 160, 320]

print("Running Experiment 0: Direct H_POVM Rademacher simulation...")
hpovm_rows = []
pbar = tqdm(total=len(d_values)*len(n_values_hpovm), desc="Exp0")
for d in d_values:
    for n in n_values_hpovm:
        seed_dn = GLOBAL_SEED + d * 1000 + n
        est_mean, est_std = hpovm_rademacher_estimate(
            d, n, HPOVM_N_SIGMA, HPOVM_STATE_REPEATS, seed_dn)
        bound = np.sqrt(d / n)
        ratio = bound / max(est_mean, EPS)
        valid = est_mean <= bound
        hpovm_rows.append(dict(d=d, n=n,
                               estimate_mean=est_mean, estimate_std=est_std,
                               bound_sqrt_d_over_n=bound,
                               ratio_bound_over_estimate=ratio,
                               valid=valid))
        pbar.set_postfix(d=d, n=n, est=f"{est_mean:.4f}", bound=f"{bound:.4f}")
        pbar.update(1)
pbar.close()

hpovm_df = pd.DataFrame(hpovm_rows)
hpovm_df.to_csv('results/hpovm_rademacher_results.csv', index=False)
print("\nH_POVM Rademacher Results:")
print(hpovm_df[['d','n','estimate_mean','estimate_std','bound_sqrt_d_over_n','valid']].to_string(index=False))
print(f"\nAll estimates below bound: {hpovm_df['valid'].all()}")

Running Experiment 0: Direct H_POVM Rademacher simulation...


Exp0:   0%|          | 0/18 [00:00<?, ?it/s]


H_POVM Rademacher Results:
 d   n  estimate_mean  estimate_std  bound_sqrt_d_over_n  valid
 4  10       0.180417      0.012916             0.632456   True
 4  20       0.124690      0.004281             0.447214   True
 4  40       0.084982      0.003355             0.316228   True
 4  80       0.064499      0.002830             0.223607   True
 4 160       0.043602      0.002216             0.158114   True
 4 320       0.031761      0.001413             0.111803   True
 8  10       0.174553      0.004493             0.894427   True
 8  20       0.123878      0.006313             0.632456   True
 8  40       0.090555      0.004785             0.447214   True
 8  80       0.060878      0.001748             0.316228   True
 8 160       0.043856      0.002953             0.223607   True
 8 320       0.032689      0.001689             0.158114   True
16  10       0.177054      0.014043             1.264911   True
16  20       0.132442      0.007832             0.894427   True
16  40      

In [4]:
# ─── Experiment 0 plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
colors = {4: '#1f77b4', 8: '#2ca02c', 16: '#d62728'}
markers = {4: 'o', 8: 's', 16: '^'}
n_arr_ref = np.array(n_values_hpovm, dtype=float)

for d in d_values:
    sub = hpovm_df[hpovm_df['d'] == d]
    ax.errorbar(sub['n'], sub['estimate_mean'], yerr=sub['estimate_std'],
                fmt=markers[d]+'-', color=colors[d], capsize=3,
                label=f'Estimate d={d}', linewidth=1.5, markersize=6, zorder=3)
    ax.plot(n_arr_ref, np.sqrt(d / n_arr_ref), '--', color=colors[d],
            alpha=0.8, label=f'$\\sqrt{{d/n}}$, d={d}')

# n^{-1/2} reference line (normalised at n=10)
ref_val = hpovm_df[hpovm_df['d']==4]['estimate_mean'].iloc[0]
ax.plot(n_arr_ref, ref_val * np.sqrt(n_arr_ref[0] / n_arr_ref),
        ':', color='grey', alpha=0.7, label='$n^{-1/2}$ ref')

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Number of states $n$', fontsize=12)
ax.set_ylabel('Empirical Rademacher complexity', fontsize=12)
ax.set_title('Experiment 0: Direct $\\mathcal{H}_{\\mathrm{POVM}}$ Rademacher simulation', fontsize=12)
ax.legend(fontsize=8, ncol=2, loc='lower left')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('figures/hpovm_rademacher_vs_n.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/hpovm_rademacher_vs_n.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/hpovm_rademacher_vs_n.{pdf,png}")

Saved: figures/hpovm_rademacher_vs_n.{pdf,png}


## 5. Dataset Generation

Nine 2D synthetic datasets for VQC experiments.

In [5]:
# ─── Dataset generation ───────────────────────────────────────────────────────

DATASET_CONFIGS = [
    ('moons_noise005',      'moons',  {'noise': 0.05}),
    ('moons_noise015',      'moons',  {'noise': 0.15}),
    ('moons_noise030',      'moons',  {'noise': 0.30}),
    ('circles_factor03',    'circles',{'noise': 0.05, 'factor': 0.3}),
    ('circles_factor06',    'circles',{'noise': 0.05, 'factor': 0.6}),
    ('classification_easy', 'classif',{'n_features':2, 'n_informative':2,
                                       'n_redundant':0, 'n_clusters_per_class':1,
                                       'class_sep':2.0}),
    ('classification_hard', 'classif',{'n_features':2, 'n_informative':2,
                                       'n_redundant':0, 'n_clusters_per_class':2,
                                       'class_sep':0.5}),
    ('blobs_tight',         'blobs',  {'n_features':2, 'cluster_std':0.5}),
    ('blobs_spread',        'blobs',  {'n_features':2, 'cluster_std':2.0}),
]
DATASET_NAMES = [c[0] for c in DATASET_CONFIGS]

def make_dataset(name, params, n_train, n_test, seed):
    """Generate a named 2D dataset, return scaled train/test splits."""
    n_total = n_train + n_test
    kind = params.get('type', None)

    # find kind from config list
    for cname, ckind, _ in DATASET_CONFIGS:
        if cname == name:
            kind = ckind
            break

    rng = np.random.RandomState(seed)
    if kind == 'moons':
        X, y = make_moons(n_samples=n_total, noise=params.get('noise',0.1),
                          random_state=seed)
    elif kind == 'circles':
        X, y = make_circles(n_samples=n_total,
                            noise=params.get('noise',0.05),
                            factor=params.get('factor',0.5),
                            random_state=seed)
    elif kind == 'classif':
        kw = {k:v for k,v in params.items() if k not in ('type',)}
        kw['n_samples'] = n_total
        kw['random_state'] = seed
        X, y = make_classification(**kw)
    elif kind == 'blobs':
        kw = {k:v for k,v in params.items() if k not in ('type',)}
        kw['n_samples'] = n_total
        kw['centers'] = 2
        kw['random_state'] = seed
        X, y = make_blobs(**kw)
        y = (y >= 1).astype(int)
    else:
        raise ValueError(f"Unknown kind {kind}")

    # ensure integer labels in {0,1}
    y = y.astype(int)
    idx = np.arange(n_total)
    rng_split = np.random.RandomState(seed + 1)
    rng_split.shuffle(idx)
    train_idx = idx[:n_train]
    test_idx  = idx[n_train:]
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_te, y_te = X[test_idx],  y[test_idx]

    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr)
    X_te = scaler.transform(X_te)
    return X_tr, y_tr, X_te, y_te

# Quick check
X_tr, y_tr, X_te, y_te = make_dataset('moons_noise015', {}, 100, 50, GLOBAL_SEED)
print(f"Dataset check: X_train={X_tr.shape}, y unique={np.unique(y_tr)}")

Dataset check: X_train=(100, 2), y unique=[0 1]


## 6 & 7. VQC Model and Training

The VQC uses RY embedding + BasicEntanglerLayers. Training uses noiseless simulator probabilities. Finite-shot effects are introduced **only during evaluation**.

In [6]:
# ─── VQC model and training ───────────────────────────────────────────────────

from pennylane import numpy as pnp
import numpy as np
import pennylane as qml


def make_noiseless_circuit(n_qubits, n_layers=2):
    """Return (qnode, weight_shape) for noiseless probability circuit."""
    dev, _ = get_device(n_qubits)

    weight_shape = qml.BasicEntanglerLayers.shape(
        n_layers=n_layers,
        n_wires=n_qubits
    )

    @qml.qnode(dev, interface="autograd")
    def circuit(weights, x):
        # Embed first two classical features into first two qubits
        qml.RY(x[0], wires=0)
        qml.RY(x[1], wires=1)

        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))

        # Returns [P(class 0), P(class 1)] for qubit 0
        return qml.probs(wires=[0])

    return circuit, weight_shape


def make_shot_circuit(n_qubits, S, n_layers=2, seed=0):
    """Return (qnode, weight_shape) for finite-shot probability circuit."""
    dev = qml.device(
        "default.qubit",
        wires=n_qubits,
        shots=S,
        seed=seed
    )

    weight_shape = qml.BasicEntanglerLayers.shape(
        n_layers=n_layers,
        n_wires=n_qubits
    )

    @qml.qnode(dev, interface="numpy")
    def circuit(weights, x):
        qml.RY(x[0], wires=0)
        qml.RY(x[1], wires=1)

        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))

        return qml.probs(wires=[0])

    return circuit, weight_shape


def squared_hinge_loss_autograd(probs_batch, y_batch):
    """
    Training surrogate loss.

    This is used only for optimisation. It is not the theorem-aligned
    evaluation metric.
    """
    p = probs_batch[:, 1]

    y_batch = pnp.array(y_batch, requires_grad=False)

    label = 2.0 * y_batch - 1.0
    score = 2.0 * p - 1.0
    margin = label * score

    loss = pnp.mean(pnp.maximum(0.0, 1.0 - margin) ** 2)
    return loss


def train_vqc(n_qubits, X_train, y_train, n_steps, lr=0.1, seed=0, n_layers=2):
    """
    Train VQC with noiseless gradients.

    This is acceptable if the paper clearly says that training uses ideal
    simulator probabilities, while finite-shot effects are evaluated afterwards.
    """
    circuit, weight_shape = make_noiseless_circuit(n_qubits, n_layers)

    rng = np.random.default_rng(seed)
    weights = pnp.array(
        rng.uniform(-np.pi, np.pi, weight_shape),
        requires_grad=True
    )

    opt = qml.GradientDescentOptimizer(stepsize=lr)

    def cost(w):
        probs = pnp.stack([
            circuit(w, x)
            for x in X_train
        ])
        return squared_hinge_loss_autograd(probs, y_train)

    final_loss = float("nan")

    for step in range(n_steps):
        weights, final_loss = opt.step_and_cost(cost, weights)

    return np.array(weights), float(final_loss)

In [7]:
# ─── Theorem-aligned score-loss evaluation ────────────────────────────────────

def score_loss_abs(p, y):
    """
    Lipschitz score loss: |p - y|.

    This is theorem-aligned:
        range: [0, 1]
        Lipschitz constant: L = 1
    """
    return np.abs(p - y)


def predict_score_noiseless(weights, n_qubits, X, n_layers=2):
    """
    Return ideal class-1 scores p_M(rho) = Tr(M rho).

    This is used as a proxy for the population score risk on a large test set.
    """
    circuit, _ = make_noiseless_circuit(n_qubits, n_layers)

    p = np.array([
        circuit(weights, x)[1]
        for x in X
    ], dtype=float)

    return p


def predict_score_shot(weights, n_qubits, X, S, n_layers=2, seed=0):
    """
    Return finite-shot class-1 score estimates p_hat_{M,S}(rho).
    """
    circuit, _ = make_shot_circuit(n_qubits, S, n_layers, seed)

    p = np.array([
        circuit(weights, x)[1]
        for x in X
    ], dtype=float)

    return p


def theorem31_score_bound(n, S, d, delta=0.05, L=1.0, exact=True):
    """
    Theorem-aligned score-risk bound.

    For absolute score loss, use L=1.

    exact=True uses the theorem constants:
        2L sqrt(d/n)
        + L sqrt(log(4n/delta)/(2S))
        + sqrt(log(2/delta)/(2n))

    exact=False uses the simplified paper/notebook version.
    """
    sample_term = 2.0 * L * np.sqrt(d / n)

    if exact:
        shot_term = L * np.sqrt(np.log(4.0 * n / delta) / (2.0 * S))
        confidence_term = np.sqrt(np.log(2.0 / delta) / (2.0 * n))
    else:
        shot_term = L * np.sqrt(np.log(2.0 * n / delta) / (2.0 * S))
        confidence_term = np.sqrt(np.log(1.0 / delta) / (2.0 * n))

    full_bound = sample_term + shot_term + confidence_term

    return {
        "sample_term": float(sample_term),
        "shot_term": float(shot_term),
        "confidence_term": float(confidence_term),
        "full_score_bound": float(full_bound),
    }


def measure_score_risk_gap(
    weights,
    n_qubits,
    X_train,
    y_train,
    X_test,
    y_test,
    S_train,
    d,
    delta=0.05,
    n_layers=2,
    seed=0,
    exact_bound=True,
):
    """
    Evaluate theorem-aligned Lipschitz score-risk gap.

    Primary theorem-aligned quantity:

        one_sided_score_gap
        = max(0, test_score_risk_ideal - train_score_risk_shot)

    where:

        train_score_risk_shot
            uses finite-shot estimates p_hat_{M,S}

        test_score_risk_ideal
            uses noiseless ideal scores p_M as a proxy for population risk

    Hard-threshold metrics are returned only as auxiliary diagnostics.
    """

    # Finite-shot empirical training score risk
    p_train_shot = predict_score_shot(
        weights=weights,
        n_qubits=n_qubits,
        X=X_train,
        S=S_train,
        n_layers=n_layers,
        seed=seed,
    )

    # Ideal/noiseless test score risk as population-risk proxy
    p_test_ideal = predict_score_noiseless(
        weights=weights,
        n_qubits=n_qubits,
        X=X_test,
        n_layers=n_layers,
    )

    # Theorem-aligned Lipschitz score risks
    train_score_risk_shot = float(
        np.mean(score_loss_abs(p_train_shot, y_train))
    )

    test_score_risk_ideal = float(
        np.mean(score_loss_abs(p_test_ideal, y_test))
    )

    one_sided_score_gap = float(
        max(0.0, test_score_risk_ideal - train_score_risk_shot)
    )

    abs_score_gap = float(
        abs(test_score_risk_ideal - train_score_risk_shot)
    )

    # Theorem bound for L = 1 absolute score loss
    bound_parts = theorem31_score_bound(
        n=len(X_train),
        S=S_train,
        d=d,
        delta=delta,
        L=1.0,
        exact=exact_bound,
    )

    valid_score_bound = bool(
        one_sided_score_gap <= bound_parts["full_score_bound"]
    )

    # Auxiliary hard-threshold metrics only
    train_hard = (p_train_shot >= 0.5).astype(int)
    test_hard = (p_test_ideal >= 0.5).astype(int)

    train_acc = float(np.mean(train_hard == y_train))
    test_acc = float(np.mean(test_hard == y_test))

    hard_acc_gap = float(abs(test_acc - train_acc))

    hard_01_risk_train = 1.0 - train_acc
    hard_01_risk_test = 1.0 - test_acc

    hard_risk_gap = float(
        max(0.0, hard_01_risk_test - hard_01_risk_train)
    )

    return {
        # Primary theorem-aligned metrics
        "train_score_risk_shot": train_score_risk_shot,
        "test_score_risk_ideal": test_score_risk_ideal,
        "one_sided_score_gap": one_sided_score_gap,
        "abs_score_gap": abs_score_gap,
        "valid_score_bound": valid_score_bound,

        # Bound decomposition
        **bound_parts,

        # Auxiliary hard-classification diagnostics
        "train_acc_shot": train_acc,
        "test_acc_ideal": test_acc,
        "hard_acc_gap": hard_acc_gap,
        "hard_risk_gap": hard_risk_gap,
    }

## 9. Generic Experiment Runner

In [8]:
# ─── Generic experiment runner ────────────────────────────────────────────────

import hashlib


def stable_dataset_seed(dataset_name, base_seed=GLOBAL_SEED):
    """
    Stable deterministic seed.

    Do not use Python's built-in hash(dataset_name), because it can change
    between Python sessions.
    """
    h = hashlib.md5(dataset_name.encode("utf-8")).hexdigest()
    offset = int(h[:8], 16) % 10000
    return int((base_seed + offset) % (2**31))


def get_dataset_params(name):
    for cname, _, params in DATASET_CONFIGS:
        if cname == name:
            return params
    return {}


def run_vqc_experiment_point(
    dataset_name,
    n_qubits,
    d,
    n_train,
    S_train,
    n_seeds,
    n_steps,
    lr,
    n_test,
    S_test_large=None,   # kept for backward compatibility, not used in theorem-aligned score evaluation
    n_layers=2,
):
    """
    Run VQC for one (dataset, d, n, S) configuration across n_seeds.

    Primary theorem-aligned metric:
        score_gap_one_sided_mean

    This is based on the bounded 1-Lipschitz score loss

        ell(p, y) = |p - y|.

    The training score risk uses finite-shot estimates.
    The test score risk uses noiseless/ideal probabilities as a proxy for population risk.

    Hard-threshold accuracy/risk metrics are auxiliary diagnostics only.
    """

    n = n_train
    S = S_train
    B = n * S

    # Theorem-aligned bound for L = 1 absolute score loss.
    bound = theorem_bound_exact(n, S, d, DELTA, L_LIPSCHITZ)
    dom = dominant_bound(n, S, d, DELTA, L_LIPSCHITZ)

    st_val = sample_term(n, d, L_LIPSCHITZ)
    sht_val = shot_term_exact(n, S, DELTA, L_LIPSCHITZ)
    ct_val = confidence_term_exact(n, DELTA)

    one_sided_gaps = []
    abs_gaps = []

    train_score_risks = []
    test_score_risks = []

    train_accs = []
    test_accs = []
    hard_acc_gaps = []
    hard_risk_gaps = []

    failures = 0

    base_seed = stable_dataset_seed(dataset_name, GLOBAL_SEED)

    for seed_idx in range(n_seeds):
        seed = int((base_seed + seed_idx * 37) % (2**31))

        # Build dataset
        params = get_dataset_params(dataset_name)

        try:
            X_tr, y_tr, X_te, y_te = make_dataset(
                dataset_name,
                params,
                n_train,
                n_test,
                seed,
            )
        except Exception as e:
            failures += 1
            continue

        # Train VQC
        try:
            weights, final_loss = train_vqc(
                n_qubits,
                X_tr,
                y_tr,
                n_steps,
                lr=lr,
                seed=seed,
                n_layers=n_layers,
            )
        except Exception as e:
            failures += 1
            continue

        # Theorem-aligned score-risk evaluation
        try:
            metrics = measure_score_risk_gap(
                weights=weights,
                n_qubits=n_qubits,
                X_train=X_tr,
                y_train=y_tr,
                X_test=X_te,
                y_test=y_te,
                S_train=S,
                d=d,
                delta=DELTA,
                n_layers=n_layers,
                seed=seed,
                exact_bound=True,
            )
        except Exception as e:
            failures += 1
            continue

        # Primary theorem-aligned score-risk quantities
        one_sided_gaps.append(metrics["one_sided_score_gap"])
        abs_gaps.append(metrics["abs_score_gap"])

        train_score_risks.append(metrics["train_score_risk_shot"])
        test_score_risks.append(metrics["test_score_risk_ideal"])

        # Auxiliary hard-threshold diagnostics
        train_accs.append(metrics["train_acc_shot"])
        test_accs.append(metrics["test_acc_ideal"])
        hard_acc_gaps.append(metrics["hard_acc_gap"])
        hard_risk_gaps.append(metrics["hard_risk_gap"])

    if len(one_sided_gaps) == 0:
        gap_mean = float("nan")
        gap_std = float("nan")
        abs_mean = float("nan")
        abs_std = float("nan")
    else:
        gap_mean = float(np.mean(one_sided_gaps))
        gap_std = float(np.std(one_sided_gaps))
        abs_mean = float(np.mean(abs_gaps))
        abs_std = float(np.std(abs_gaps))

    valid = (not np.isnan(gap_mean)) and (gap_mean <= bound)
    slack = bound - gap_mean if not np.isnan(gap_mean) else float("nan")
    tight = safe_tightness_ratio(bound, gap_mean, TIGHTNESS_GAP_MIN)

    return dict(
        dataset=dataset_name,
        d=d,
        n_qubits=n_qubits,
        n=n,
        S=S,
        B=B,
        n_seeds=n_seeds,
        n_successful_seeds=len(one_sided_gaps),
        n_failed_seeds=failures,

        # Primary theorem-aligned metrics
        score_gap_one_sided_mean=gap_mean,
        score_gap_one_sided_std=gap_std,
        score_gap_abs_mean=abs_mean,
        score_gap_abs_std=abs_std,

        train_score_risk_shot_mean=(
            float(np.nanmean(train_score_risks))
            if train_score_risks else float("nan")
        ),
        test_score_risk_ideal_mean=(
            float(np.nanmean(test_score_risks))
            if test_score_risks else float("nan")
        ),

        # Auxiliary hard-threshold diagnostics only
        train_acc_shot_mean=(
            float(np.nanmean(train_accs))
            if train_accs else float("nan")
        ),
        test_acc_ideal_mean=(
            float(np.nanmean(test_accs))
            if test_accs else float("nan")
        ),
        hard_acc_gap_mean=(
            float(np.nanmean(hard_acc_gaps))
            if hard_acc_gaps else float("nan")
        ),
        hard_risk_gap_mean=(
            float(np.nanmean(hard_risk_gaps))
            if hard_risk_gaps else float("nan")
        ),

        # Bound decomposition
        sample_term=st_val,
        shot_term=sht_val,
        confidence_term=ct_val,
        full_score_bound=bound,
        dominant_score_bound=dom,

        # Validity / slack / tightness
        valid_score_gap_full_bound=valid,
        slack_score=slack,
        tightness_ratio_score=tight,
    )


print("Generic theorem-aligned VQC runner defined.")

Generic theorem-aligned VQC runner defined.


## 10. Experiment 1: Theorem-Aligned Score-Risk Gap vs n

Fixed S=100, sweep n over [10, 20, 40, 80, 160].

In [ ]:
# ─── Experiment 1: score-risk gap vs n ───────────────────────────────────────
S_EXP1 = 100
n_grid_exp1 = [10, 20, 40, 80, 160]
if SMOKE_TEST:
    n_grid_exp1 = [10, 40]

lr_exp1 = 0.05

print("Running Experiment 1: score-risk gap vs training examples n ...")
exp1_rows = []
n_qubits_list = [(2, 4), (4, 16)]
total_tasks = len(DATASET_NAMES) * len(n_qubits_list) * len(n_grid_exp1)
pbar = tqdm(total=total_tasks, desc="Exp1")

for n_qubits, d in n_qubits_list:
    n_seeds = EXP1_SEEDS[d]
    for n in n_grid_exp1:
        for ds_name in DATASET_NAMES:
            row = run_vqc_experiment_point(
                dataset_name=ds_name,
                n_qubits=n_qubits, d=d,
                n_train=n, S_train=S_EXP1,
                n_seeds=n_seeds, n_steps=N_STEPS, lr=lr_exp1,
                n_test=N_TEST_LARGE, S_test_large=S_TEST_LARGE,
            )
            exp1_rows.append(row)
            pbar.set_postfix(d=d, n=n, ds=ds_name[:8],
                             gap=f"{row['score_gap_one_sided_mean']:.4f}",
                             valid=row['valid_score_gap_full_bound'])
            pbar.update(1)
pbar.close()

exp1_df = pd.DataFrame(exp1_rows)
exp1_df.to_csv('results/exp1_score_gap_vs_n.csv', index=False)

validity_exp1 = exp1_df['valid_score_gap_full_bound'].all()
print(f"\nExp1 all gaps below bound: {validity_exp1}")
print(exp1_df[['d','n','dataset','score_gap_one_sided_mean',
               'full_score_bound','valid_score_gap_full_bound']].to_string(index=False))

Running Experiment 1: score-risk gap vs training examples n ...


Exp1:   0%|          | 0/90 [00:00<?, ?it/s]


Exp1 all gaps below bound: True


KeyError: "['full_bound'] not in index"

In [10]:
# ─── Experiment 1 plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

cmap = plt.cm.tab20
ds_colors = {name: cmap(i/len(DATASET_NAMES)) for i, name in enumerate(DATASET_NAMES)}
n_arr = np.array(n_grid_exp1, dtype=float)

for ax_idx, (n_qubits, d) in enumerate([(2,4),(4,16)]):
    ax = axes[ax_idx]
    sub_d = exp1_df[exp1_df['d'] == d]

    for ds_name in DATASET_NAMES:
        sub = sub_d[sub_d['dataset'] == ds_name].sort_values('n')
        if sub.empty:
            continue
        gaps = sub['score_gap_one_sided_mean'].values
        stds = sub['score_gap_one_sided_std'].values
        ns   = sub['n'].values.astype(float)
        ax.errorbar(ns, gaps + EPS, yerr=stds, fmt='o-',
                    color=ds_colors[ds_name], alpha=0.7,
                    markersize=4, linewidth=1, capsize=2,
                    label=ds_name.replace('_',' '))

    # theorem bound
    bounds_arr = [theorem_bound_exact(n, S_EXP1, d, DELTA) for n in n_grid_exp1]
    ax.plot(n_arr, bounds_arr, 'k--', linewidth=2, label='Full Theorem 3.1 bound')

    # n^{-1/2} reference
    ref_start = bounds_arr[0]
    ax.plot(n_arr, ref_start * np.sqrt(n_arr[0] / n_arr), ':',
            color='gray', linewidth=1.5, label='$n^{-1/2}$ ref')

    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('Number of training examples $n$', fontsize=11)
    ax.set_ylabel('One-sided score-risk gap', fontsize=11)
    ax.set_title(f'Experiment 1 (d={d}, {n_qubits} qubits)', fontsize=11)
    ax.legend(fontsize=6, ncol=2, loc='lower left')
    ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Experiment 1: Lipschitz score-risk gap vs training examples', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exp1_score_gap_vs_n.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/exp1_score_gap_vs_n.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/exp1_score_gap_vs_n.{pdf,png}")

# ─── Slope regression ─────────────────────────────────────────────────────────
slope_rows = []
for n_qubits, d in [(2,4),(4,16)]:
    sub_d = exp1_df[exp1_df['d'] == d]
    for ds_name in DATASET_NAMES:
        sub = sub_d[sub_d['dataset'] == ds_name].sort_values('n')
        if sub.shape[0] < 3:
            continue
        log_n = np.log(sub['n'].values.astype(float))
        log_g = np.log(sub['score_gap_one_sided_mean'].values + EPS)
        res = stats.linregress(log_n, log_g)
        slope_rows.append(dict(d=d, dataset=ds_name, slope=res.slope,
                               stderr=res.stderr, rvalue=res.rvalue,
                               pvalue=res.pvalue))
slope_df = pd.DataFrame(slope_rows)
slope_df.to_csv('tables/exp1_slope_regression.csv', index=False)

summary_rows = []
for d in [4, 16]:
    sub = slope_df[slope_df['d']==d]
    summary_rows.append(dict(d=d, n_datasets=len(sub),
                             mean_slope=round(sub['slope'].mean(),3),
                             std_slope=round(sub['slope'].std(),3)))
slope_summary_df = pd.DataFrame(summary_rows)
slope_summary_df.to_csv('tables/exp1_slope_summary.csv', index=False)
print("\nSlope summary (predicted: -0.5):")
print(slope_summary_df.to_string(index=False))

Saved: figures/exp1_score_gap_vs_n.{pdf,png}

Slope summary (predicted: -0.5):
 d  n_datasets  mean_slope  std_slope
 4           9      -0.607      0.371
16           9      -1.216      1.365


## 11. Experiment 2: Score-Risk Gap vs Shots S

Fixed n=80, sweep S over [10, 25, 50, 100, 250, 500].

The theorem shot term decreases as S^{-1/2}. Empirical VQC gaps may flatten because tasks are structured and not purely shot-limited.

In [11]:
# ─── Experiment 2: score-risk gap vs shots S ─────────────────────────────────
N_EXP2 = 80
S_grid_exp2 = [10, 25, 50, 100, 250, 500]
if SMOKE_TEST:
    S_grid_exp2 = [10, 100]

lr_exp2 = 0.05

print("Running Experiment 2: score-risk gap vs shots S ...")
exp2_rows = []
total_tasks = len(DATASET_NAMES) * len(n_qubits_list) * len(S_grid_exp2)
pbar = tqdm(total=total_tasks, desc="Exp2")

for n_qubits, d in n_qubits_list:
    n_seeds = EXP2_SEEDS[d]
    for S in S_grid_exp2:
        for ds_name in DATASET_NAMES:
            row = run_vqc_experiment_point(
                dataset_name=ds_name,
                n_qubits=n_qubits, d=d,
                n_train=N_EXP2, S_train=S,
                n_seeds=n_seeds, n_steps=N_STEPS, lr=lr_exp2,
                n_test=N_TEST_LARGE, S_test_large=S_TEST_LARGE,
            )
            exp2_rows.append(row)
            pbar.set_postfix(d=d, S=S, ds=ds_name[:8],
                             gap=f"{row['score_gap_one_sided_mean']:.4f}")
            pbar.update(1)
pbar.close()

exp2_df = pd.DataFrame(exp2_rows)
exp2_df.to_csv('results/exp2_score_gap_vs_S.csv', index=False)

validity_exp2 = exp2_df['valid_score_gap_full_bound'].all()
print(f"\nExp2 all gaps below bound: {validity_exp2}")

Running Experiment 2: score-risk gap vs shots S ...


Exp2:   0%|          | 0/108 [00:00<?, ?it/s]


Exp2 all gaps below bound: True


In [12]:
# ─── Experiment 2 plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
S_arr = np.array(S_grid_exp2, dtype=float)

for ax_idx, (n_qubits, d) in enumerate([(2,4),(4,16)]):
    ax = axes[ax_idx]
    sub_d = exp2_df[exp2_df['d'] == d]

    for ds_name in DATASET_NAMES:
        sub = sub_d[sub_d['dataset'] == ds_name].sort_values('S')
        if sub.empty:
            continue
        gaps = sub['score_gap_one_sided_mean'].values
        stds = sub['score_gap_one_sided_std'].values
        Ss   = sub['S'].values.astype(float)
        ax.errorbar(Ss, gaps + EPS, yerr=stds, fmt='o-',
                    color=ds_colors[ds_name], alpha=0.7,
                    markersize=4, linewidth=1, capsize=2,
                    label=ds_name.replace('_',' '))

    bounds_arr = [theorem_bound_exact(N_EXP2, S, d, DELTA) for S in S_grid_exp2]
    ax.plot(S_arr, bounds_arr, 'k--', linewidth=2, label='Full Theorem 3.1 bound')

    # S^{-1/2} reference
    ref_start = bounds_arr[0]
    ax.plot(S_arr, ref_start * np.sqrt(S_arr[0] / S_arr), ':',
            color='gray', linewidth=1.5, label='$S^{-1/2}$ ref')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Shots per example $S$', fontsize=11)
    ax.set_ylabel('One-sided score-risk gap', fontsize=11)
    ax.set_title(f'Experiment 2 (d={d}, {n_qubits} qubits)', fontsize=11)
    ax.legend(fontsize=6, ncol=2, loc='lower left')
    ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Experiment 2: Lipschitz score-risk gap vs shots per example', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exp2_score_gap_vs_S.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/exp2_score_gap_vs_S.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/exp2_score_gap_vs_S.{pdf,png}")

Saved: figures/exp2_score_gap_vs_S.{pdf,png}


## 12. Experiment 3: Fixed-Budget Shot–Sample Allocation

Under fixed budget B = n·S:
- **Small n** → large sample term 2√(d/n) dominates → **sample-limited / data-limited**
- **Large n** → S=B/n is small, shot term dominates → **shot-limited**

In [13]:
# ─── Experiment 3: fixed-budget shot-sample allocation ───────────────────────
B_grid_exp3 = [500, 1000, 2000]
if SMOKE_TEST:
    B_grid_exp3 = [500]

lr_exp3 = 0.05

def make_n_grid_for_budget(B, d, n_min=5, n_max=None, n_points=8):
    """Log-spaced n grid around n*, clipped to valid range."""
    n_star = n_star_approx(B, d, DELTA)
    if n_max is None:
        n_max = min(B // 5, 200)
    n_lo = max(n_min, int(n_star / 4))
    n_hi = min(n_max, int(n_star * 4))
    ns = np.unique(np.round(np.logspace(
        np.log10(n_lo), np.log10(n_hi), n_points)).astype(int))
    ns = ns[ns >= n_min]
    # ensure S>=5
    ns = ns[B // ns >= 5]
    return ns.tolist()

print("Running Experiment 3: fixed-budget allocation ...")
exp3_rows = []
n_seeds_exp3 = EXP3_SEEDS
total_tasks = len(B_grid_exp3) * len(n_qubits_list)
for B in B_grid_exp3:
    for n_qubits, d in n_qubits_list:
        n_grid = make_n_grid_for_budget(B, d)
        print(f"  B={B}, d={d}: n_grid={n_grid}")
        pbar = tqdm(total=len(n_grid)*len(DATASET_NAMES), desc=f"Exp3 B={B} d={d}")
        for n in n_grid:
            S = max(5, B // n)
            for ds_name in DATASET_NAMES:
                row = run_vqc_experiment_point(
                    dataset_name=ds_name,
                    n_qubits=n_qubits, d=d,
                    n_train=n, S_train=S,
                    n_seeds=n_seeds_exp3, n_steps=N_STEPS, lr=lr_exp3,
                    n_test=N_TEST_LARGE, S_test_large=S_TEST_LARGE,
                )
                row['B_budget'] = B
                exp3_rows.append(row)
                pbar.update(1)
        pbar.close()

exp3_df = pd.DataFrame(exp3_rows)
exp3_df.to_csv('results/exp3_fixed_budget.csv', index=False)
print(f"\nExp3 all gaps below bound: {exp3_df['valid_score_gap_full_bound'].all()}")
print(f"Total rows: {len(exp3_df)}")

Running Experiment 3: fixed-budget allocation ...
  B=500, d=4: n_grid=[10, 14, 19, 27, 37, 52, 72, 100]


Exp3 B=500 d=4:   0%|          | 0/72 [00:00<?, ?it/s]

  B=500, d=16: n_grid=[20, 25, 32, 40, 50, 63, 79, 100]


Exp3 B=500 d=16:   0%|          | 0/72 [00:00<?, ?it/s]

  B=1000, d=4: n_grid=[13, 19, 28, 42, 62, 92, 135, 200]


...

  B=2000, d=4: n_grid=[18, 25, 36, 51, 71, 101, 142, 200]


Exp3 B=2000 d=4:   0%|          | 0/72 [00:00<?, ?it/s]

  B=2000, d=16: n_grid=[37, 47, 60, 76, 97, 123, 157, 200]


Exp3 B=2000 d=16:   0%|          | 0/72 [00:00<?, ?it/s]


Exp3 all gaps below bound: True
Total rows: 432


In [14]:
# ─── Experiment 3: grid figure ────────────────────────────────────────────────
n_rows_plot = len(B_grid_exp3)
n_cols_plot = 2
fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                         figsize=(14, 4.5 * n_rows_plot), squeeze=False)

for row_idx, B in enumerate(B_grid_exp3):
    for col_idx, (n_qubits, d) in enumerate([(2,4),(4,16)]):
        ax = axes[row_idx][col_idx]
        sub_bd = exp3_df[(exp3_df['B_budget']==B) & (exp3_df['d']==d)]
        n_grid_plot = sorted(sub_bd['n'].unique())
        n_arr_plot  = np.array(n_grid_plot, dtype=float)

        for ds_name in DATASET_NAMES:
            sub = sub_bd[sub_bd['dataset']==ds_name].sort_values('n')
            if sub.empty:
                continue
            ax.plot(sub['n'].values, sub['score_gap_one_sided_mean'].values + EPS,
                    'o-', color=ds_colors[ds_name], alpha=0.7,
                    markersize=3, linewidth=1,
                    label=ds_name.replace('_',' '))

        # theorem bound with integer S = B//n
        if len(n_arr_plot) > 0:
            bounds_plot = [theorem_bound_exact(n, max(5,B//n), d, DELTA)
                           for n in n_arr_plot]
            ax.plot(n_arr_plot, bounds_plot, 'k--', linewidth=2,
                    label='Full bound')

        # n* vertical line
        n_star = n_star_approx(B, d, DELTA)
        ax.axvline(n_star, color='k', linestyle=':', linewidth=1.5,
                   label=f'$n^*={n_star:.0f}$')

        # regime annotations
        n_mid = np.sqrt(n_arr_plot[0] * n_arr_plot[-1]) if len(n_arr_plot) > 0 else n_star
        ax.text(0.08, 0.88, 'sample-limited\n(data-limited)',
                transform=ax.transAxes, fontsize=7, color='navy',
                ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.7))
        ax.text(0.72, 0.88, 'shot-limited',
                transform=ax.transAxes, fontsize=7, color='darkred',
                ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.7))

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('$n$', fontsize=10)
        ax.set_ylabel('One-sided score-risk gap', fontsize=9)
        ax.set_title(f'B={B}, d={d}', fontsize=10)
        ax.grid(True, alpha=0.3, which='both')
        if row_idx == 0 and col_idx == 1:
            ax.legend(fontsize=5, ncol=2, loc='upper right')

plt.suptitle('Experiment 3: Fixed-budget shot–sample allocation\n'
             '(Primary metric: Lipschitz score-risk gap)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exp3_fixed_budget_grid.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/exp3_fixed_budget_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/exp3_fixed_budget_grid.{pdf,png}")

# ─── Theoretical tradeoff plot with correct regime labels ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
B_theory = 1000
n_fine = np.logspace(np.log10(5), np.log10(B_theory // 2), 300)

for ax_idx, (n_qubits, d) in enumerate([(2,4),(4,16)]):
    ax = axes[ax_idx]
    G_vals = G_tradeoff(n_fine, B_theory, d, DELTA)
    ax.plot(n_fine, G_vals, 'b-', linewidth=2, label=f'$\\mathcal{{G}}(n)$ bound')

    n_star = n_star_approx(B_theory, d, DELTA)
    ax.axvline(n_star, color='k', linestyle='--', linewidth=1.5,
               label=f'$n^*={n_star:.0f}$')

    # Shade regimes
    ax.axvspan(n_fine[0], n_star, alpha=0.08, color='navy',
               label='sample-limited (data-limited)')
    ax.axvspan(n_star, n_fine[-1], alpha=0.08, color='darkred',
               label='shot-limited')
    ax.text(n_fine[0]*1.1, np.min(G_vals)*1.5, 'sample-limited\n(data-limited)',
            fontsize=9, color='navy')
    ax.text(n_star*1.1, np.min(G_vals)*1.5, 'shot-limited',
            fontsize=9, color='darkred')

    ax.set_xscale('log')
    ax.set_xlabel('$n$ (number of training examples)', fontsize=11)
    ax.set_ylabel('$\\mathcal{G}(n)$ bound', fontsize=11)
    ax.set_title(f'Theoretical tradeoff: B={B_theory}, d={d}', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Theoretical shot–sample tradeoff with correct regime labels\n'
             'small n = sample-limited/data-limited; large n = shot-limited', fontsize=11)
plt.tight_layout()
plt.savefig('figures/theoretical_tradeoff_correct_regimes.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/theoretical_tradeoff_correct_regimes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/theoretical_tradeoff_correct_regimes.{pdf,png}")

Saved: figures/exp3_fixed_budget_grid.{pdf,png}
Saved: figures/theoretical_tradeoff_correct_regimes.{pdf,png}


## 13. Experiment 4: Tightness and Slack at Predicted Allocation

Large tightness ratios are **expected**: the theorem is distribution-free and worst-case over the full H_POVM class, while VQC uses a restricted class on structured datasets. Additive slack is the more stable diagnostic.

In [15]:
# ─── Experiment 4: tightness at predicted allocation ─────────────────────────
B_grid_exp4 = [500, 1000, 2000]
if SMOKE_TEST:
    B_grid_exp4 = [500]

n_seeds_exp4 = EXP4_SEEDS
lr_exp4 = 0.05

print("Running Experiment 4: tightness at predicted allocation ...")
exp4_rows = []
total_tasks4 = len(B_grid_exp4) * len(n_qubits_list) * len(DATASET_NAMES)
pbar = tqdm(total=total_tasks4, desc="Exp4")

for B in B_grid_exp4:
    for n_qubits, d in n_qubits_list:
        n_alloc, S_alloc = nearest_integer_allocation(B, d, DELTA)
        for ds_name in DATASET_NAMES:
            row = run_vqc_experiment_point(
                dataset_name=ds_name,
                n_qubits=n_qubits, d=d,
                n_train=n_alloc, S_train=S_alloc,
                n_seeds=n_seeds_exp4, n_steps=N_STEPS, lr=lr_exp4,
                n_test=N_TEST_LARGE, S_test_large=S_TEST_LARGE,
            )
            row['B_budget'] = B
            row['n_alloc'] = n_alloc
            row['S_alloc'] = S_alloc
            exp4_rows.append(row)
            pbar.update(1)
pbar.close()

exp4_df = pd.DataFrame(exp4_rows)
exp4_df.to_csv('results/exp4_tightness_at_allocation.csv', index=False)
print(f"\nExp4 all gaps below bound: {exp4_df['valid_score_gap_full_bound'].all()}")

# ─── Summary table ────────────────────────────────────────────────────────────
tight_rows = []
for B in B_grid_exp4:
    for n_qubits, d in n_qubits_list:
        n_alloc, S_alloc = nearest_integer_allocation(B, d, DELTA)
        sub = exp4_df[(exp4_df['B_budget']==B) & (exp4_df['d']==d)]
        if sub.empty:
            continue
        bound_val = theorem_bound_exact(n_alloc, S_alloc, d, DELTA)
        gaps = sub['score_gap_one_sided_mean'].dropna()
        slack_vals = sub['slack_score'].dropna()
        tight_vals = sub['tightness_ratio_score'].dropna()
        tight_vals_thresh = tight_vals[sub.loc[tight_vals.index,'score_gap_one_sided_mean'] >= TIGHTNESS_GAP_MIN]
        n_reported = len(tight_vals_thresh)
        n_total = len(sub)
        tight_rows.append(dict(
            B=B, d=d, n_alloc=n_alloc, S_alloc=S_alloc,
            bound=round(bound_val,4),
            mean_gap=round(gaps.mean(),4) if len(gaps)>0 else float('nan'),
            mean_slack=round(slack_vals.mean(),4) if len(slack_vals)>0 else float('nan'),
            mean_tightness=round(tight_vals_thresh.mean(),2) if len(tight_vals_thresh)>0 else float('nan'),
            n_reported=f"{n_reported}/{n_total}",
        ))
tight_summary_df = pd.DataFrame(tight_rows)
tight_summary_df.to_csv('tables/exp4_tightness_summary.csv', index=False)
print("\nExp4 tightness summary:")
print(tight_summary_df.to_string(index=False))

Running Experiment 4: tightness at predicted allocation ...


Exp4:   0%|          | 0/54 [00:00<?, ?it/s]


Exp4 all gaps below bound: True

Exp4 tightness summary:
   B  d  n_alloc  S_alloc  bound  mean_gap  mean_slack  mean_tightness n_reported
 500  4       40       12 1.4271    0.0203      1.4068           94.15        9/9
 500 16       80        6 1.9009    0.0102      1.8907          268.40        9/9
1000  4       55       18 1.2052    0.0162      1.1890          128.44        9/9
1000 16      110        9 1.6026    0.0061      1.5966          329.41        8/9
2000  4       75       26 1.0277    0.0096      1.0181          181.22        9/9
2000 16      151       13 1.3628    0.0047      1.3581          463.29        9/9


In [17]:
# ─── Experiment 4 plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

configs = [(B, d) for B in B_grid_exp4 for _, d in n_qubits_list]
x_pos = np.arange(len(configs))
bar_width = 0.35
tight_one = []
tight_abs = []
x_labels  = []
n_reported_labels = []

for B, d in configs:
    sub = exp4_df[(exp4_df['B_budget']==B) & (exp4_df['d']==d)]
    valid_sub = sub[sub['score_gap_one_sided_mean'] >= TIGHTNESS_GAP_MIN]
    n_rep = len(valid_sub)
    n_tot = len(sub)
    if len(valid_sub) > 0:
        tight_one.append(valid_sub['tightness_ratio_score'].mean())
    else:
        tight_one.append(0)
    # absolute gap tightness
    valid_abs = sub[sub['score_gap_abs_mean'] >= TIGHTNESS_GAP_MIN]
    if len(valid_abs) > 0:
        r = (valid_abs['full_score_bound'] / valid_abs['score_gap_abs_mean']).mean()
        tight_abs.append(r)
    else:
        tight_abs.append(0)
    x_labels.append(f'B={B}\nd={d}')
    n_reported_labels.append(f'{n_rep}/{n_tot}')

bars1 = ax.bar(x_pos - bar_width/2, tight_one, bar_width,
               label='bound/one-sided gap (gap>=0.001)',
               color='steelblue', alpha=0.8)
bars2 = ax.bar(x_pos + bar_width/2, tight_abs, bar_width,
               label='bound/abs gap (gap>=0.001)',
               color='orange', alpha=0.8)

# annotate with n_reported
for i, (b1, b2, lbl) in enumerate(zip(bars1, bars2, n_reported_labels)):
    h = max(b1.get_height(), b2.get_height())
    ax.text(i, h + 1, lbl, ha='center', va='bottom', fontsize=7)

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, fontsize=9)
ax.set_ylabel('Tightness ratio (bound / empirical gap)', fontsize=10)
ax.set_title('Experiment 4: Thresholded tightness ratios at predicted allocation $(n^*, S^*)$\n'
             '(Large ratios expected: worst-case bound vs restricted VQC class)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figures/exp4_thresholded_tightness.pdf', dpi=150, bbox_inches='tight')
plt.savefig('figures/exp4_thresholded_tightness.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/exp4_thresholded_tightness.{pdf,png}")

Saved: figures/exp4_thresholded_tightness.{pdf,png}


## 14. Summary Tables and Final Validation

In [19]:
# ─── Combine all VQC results ──────────────────────────────────────────────────
all_vqc = pd.concat([
    exp1_df.assign(experiment='exp1'),
    exp2_df.assign(experiment='exp2'),
    exp3_df.assign(experiment='exp3'),
    exp4_df.assign(experiment='exp4'),
], ignore_index=True)
all_vqc.to_csv('results/all_vqc_results_combined.csv', index=False)

# ─── Validity summary ─────────────────────────────────────────────────────────
validity_rows = []
for exp_name in ['exp1','exp2','exp3','exp4']:
    sub = all_vqc[all_vqc['experiment']==exp_name]
    for d in [4, 16]:
        subd = sub[sub['d']==d]
        if subd.empty:
            continue
        valid_frac = subd['valid_score_gap_full_bound'].mean()
        thresh_sub = subd[subd['score_gap_one_sided_mean'] >= TIGHTNESS_GAP_MIN]
        validity_rows.append(dict(
            experiment=exp_name, d=d,
            num_points=len(subd),
            validity_fraction=round(valid_frac,4),
            max_score_gap=round(subd['score_gap_one_sided_mean'].max(),4),
            min_full_bound=round(subd['full_score_bound'].min(),4),
            mean_slack=round(subd['slack_score'].mean(),4),
            median_tightness_reported=round(thresh_sub['tightness_ratio_score'].median(),2) if len(thresh_sub)>0 else float('nan'),
            num_tightness_reported=len(thresh_sub),
            num_tightness_total=len(subd),
        ))

validity_df = pd.DataFrame(validity_rows)
validity_df.to_csv('tables/validity_summary.csv', index=False)
print("Validity summary:")
print(validity_df.to_string(index=False))

# ─── Final boolean checks ─────────────────────────────────────────────────────
hpovm_all_valid = hpovm_df['valid'].all()
vqc_all_valid   = all_vqc['valid_score_gap_full_bound'].all()

print("\n" + "="*60)
print("FINAL VALIDATION CHECKS")
print("="*60)
print(f"[{'PASS' if hpovm_all_valid else 'FAIL'}] All H_POVM estimates below sqrt(d/n)")
print(f"[{'PASS' if vqc_all_valid   else 'FAIL'}] All theorem-aligned VQC score gaps below full bound")
print(f"[PASS] Primary theorem-validity check uses Lipschitz score loss |p-y|, NOT hard 0/1 loss")
print(f"[PASS] Exact theorem constants used: log(4n/delta)/(2S) and log(2/delta)/(2n)")
print("="*60)
print(f"\nRegime convention:")
print(f"  small n (n << n*): sample-limited / data-limited  [2*sqrt(d/n) dominates]")
print(f"  large n (n >> n*): shot-limited               [sqrt(log(4n/delta)/(2S)) dominates]")

Validity summary:
experiment  d  num_points  validity_fraction  max_score_gap  min_full_bound  mean_slack  median_tightness_reported  num_tightness_reported  num_tightness_total
      exp1  4          45                1.0         0.0717          0.6410      1.1365                      81.81                      45                   45
      exp1 16          45                1.0         0.0259          0.9573      1.8579                     336.32                      38                   45
      exp2  4          54                1.0         0.0369          0.6927      0.8901                     100.49                      54                   54
      exp2 16          54                1.0         0.0114          1.1399      1.3444                     307.04                      40                   54
      exp3  4         216                1.0         0.0690          1.0197      1.3404                      92.05                     207                  216
      exp3 16         

## 15. Compatibility with the Manuscript

**Direct H_POVM experiment (Experiment 0):**
- Implements the exact positive-eigenvalue supremum from the proof of Proposition 3.1.
- This is the most theorem-aligned experiment.

**VQC Experiments (1–4):**
- Use a restricted variational circuit class with RY embedding + BasicEntanglerLayers.
- Should **not** be described as worst-case validation of the full H_POVM class.
- The primary VQC metric is the **one-sided Lipschitz score-risk gap** using ℓ(p,y) = |p−y|.
- Hard accuracy is auxiliary only.

**Exact theorem constants used:**
- Shot term: $L\sqrt{\log(4n/\delta)/(2S)}$
- Confidence term: $\sqrt{\log(2/\delta)/(2n)}$

**Correct fixed-budget regime labels (under B = nS):**
- **small n (n ≪ n\*):** sample-limited / data-limited (sample term $2\sqrt{d/n}$ dominates)
- **large n (n ≫ n\*):** shot-limited (shot term $\sqrt{\log(4n/\delta)/(2S)}$ dominates)

**d=16 VQC setting:**
- Uses the same two input features embedded into the first two qubits, with remaining qubits participating through entangling layers.
- This should be interpreted as a conservative larger-Hilbert-space illustration, **not** a clean empirical test of $\sqrt{d}$ scaling.